In [ ]:
# !pip install mamba-ssm tilelang quack-kernels --no-build-isolation
# make sure all requirements are downloaded

In [ ]:
import os
import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from itertools import permutations

def calc_si_sdr(estimate, target, epsilon=1e-8):
    target = target - torch.mean(target, dim=-1, keepdim=True)
    estimate = estimate - torch.mean(estimate, dim=-1, keepdim=True)
    target_energy = torch.sum(target ** 2, dim=-1, keepdim=True) + epsilon
    dot_product = torch.sum(target * estimate, dim=-1, keepdim=True)
    alpha = dot_product / target_energy
    target_scaled = alpha * target
    noise = estimate - target_scaled
    signal_energy = torch.sum(target_scaled ** 2, dim=-1) + epsilon
    noise_energy = torch.sum(noise ** 2, dim=-1) + epsilon
    ratio = torch.clamp(signal_energy / noise_energy, min=epsilon)
    si_sdr = 10 * torch.log10(ratio)
    return -si_sdr

def or_pit_loss(est_speaker, est_residual, targets):
    batch_size = targets.shape[0]
    num_speakers = targets.shape[1] 
    
    # Split targets back into individual speakers
    target_speakers = torch.chunk(targets, num_speakers, dim=1)
    
    best_losses = []
    best_indices = []
    
    for b in range(batch_size):
        b_est_spk = est_speaker[b]
        b_est_res = est_residual[b]
        b_targets = [t[b].squeeze(0) for t in target_speakers]
        
        best_loss_tensor = None
        best_idx = 0
        
        for i in range(num_speakers):
            loss_spk = calc_si_sdr(b_est_spk.unsqueeze(0), b_targets[i].unsqueeze(0))
            
            res_target = sum([b_targets[j] for j in range(num_speakers) if j != i])
            loss_res = calc_si_sdr(b_est_res.unsqueeze(0), res_target.unsqueeze(0))
            
            total_loss = loss_spk + loss_res
            
            # If this is the first iteration, or we found a better (lower) loss
            if best_loss_tensor is None or total_loss < best_loss_tensor:
                best_loss_tensor = total_loss
                best_idx = i
                
        best_losses.append(best_loss_tensor)
        best_indices.append(best_idx)
        
    return torch.stack(best_losses).mean(), best_indices

In [ ]:
from mamba_ssm import Mamba

class MambaDeflationaryExtractor(nn.Module):
    def __init__(self, d_model=256, d_state=16, d_conv=4, expand=2, num_blocks=4):
        super().__init__()
        
        # Encoder: Audio waveform to latent features (1D Conv)
        self.encoder = nn.Conv1d(in_channels=1, out_channels=d_model, kernel_size=16, stride=8, padding=4)
        
        # Stacked Mamba Blocks (efficient sequential processing)
        self.mamba_blocks = nn.ModuleList([
            Mamba(
                d_model=d_model, # Model dimension
                d_state=d_state, # State dimension (controls memory compression)
                d_conv=d_conv,   # Local convolution width
                expand=expand    # Block expansion factor
            ) for _ in range(num_blocks)
        ])
        
        # Dual Decoders: 
        # Decoder A isolates the dominant speaker
        self.speaker_decoder = nn.ConvTranspose1d(in_channels=d_model, out_channels=1, kernel_size=16, stride=8, padding=4)
        # Decoder B outputs the rest of the mixture
        self.residual_decoder = nn.ConvTranspose1d(in_channels=d_model, out_channels=1, kernel_size=16, stride=8, padding=4)

    def forward(self, x):
        """
        x shape: (batch_size, 1, sequence_length)
        """
        # Encode
        encoded = self.encoder(x) # Shape: (batch, d_model, seq_len_latent)
        
        # Mamba expects shape (batch, seq_len, d_model), so we transpose
        latent = encoded.transpose(1, 2)
        
        # Pass through Mamba backbone
        for block in self.mamba_blocks:
            latent = block(latent)
            
        # Transpose back for decoding
        latent = latent.transpose(1, 2) # Shape: (batch, d_model, seq_len_latent)
        
        # Decode into two separate audio signals
        est_speaker = self.speaker_decoder(latent)
        est_residual = self.residual_decoder(latent)
        
        # Squeeze out the channel dimension for loss calculation
        return est_speaker.squeeze(1), est_residual.squeeze(1)

# dataset with automatic padding 
import os
import torch
import torchaudio
from torch.utils.data import Dataset

class CustomSpeechMixtureDataset(Dataset):
    def __init__(self, base_dir, num_speakers, fixed_length=80000): 
        # fixed_length: 80000 is 5 seconds at 16kHz.
        self.base_dir = base_dir
        self.num_speakers = num_speakers
        self.fixed_length = fixed_length
        
        self.mix_folders = sorted([
            f for f in os.listdir(base_dir) 
            if os.path.isdir(os.path.join(base_dir, f)) and f.startswith('mix_')
        ])

    def __len__(self):
        return len(self.mix_folders)

    def __getitem__(self, idx):
        folder_name = self.mix_folders[idx]
        folder_path = os.path.join(self.base_dir, folder_name)
        
        # 1. Load and Pad/Truncate Mixture
        mix_path = os.path.join(folder_path, "mix.wav")
        mixture, _ = torchaudio.load(mix_path)
        mixture = self._pad_or_truncate(mixture)
        
        # 2. Load and Pad/Truncate Sources
        sources = []
        for i in range(1, self.num_speakers + 1):
            source_path = os.path.join(folder_path, f"s{i}.wav")
            source_audio, _ = torchaudio.load(source_path)
            source_audio = self._pad_or_truncate(source_audio)
            sources.append(source_audio)
            
        target_sources = torch.cat(sources, dim=0)
        return mixture, target_sources

    def _pad_or_truncate(self, audio):
        """Forces audio to be exactly fixed_length."""
        length = audio.shape[-1]
        if length > self.fixed_length:
            return audio[:, :self.fixed_length]  # Truncate
        else:
            padding = self.fixed_length - length
            return torch.nn.functional.pad(audio, (0, padding))  # Pad with zeros

In [ ]:
import os
import torch
from torch.utils.data import DataLoader

def evaluate_validation_set(model, val_loader, device):
    model.eval()
    total_si_sdr = 0.0
    valid_batches = 0 # Keep track of healthy batches
    
    with torch.no_grad(): 
        for mixture, targets in val_loader:
            mixture = mixture.to(device)
            targets = targets.to(device)
            
            est_speaker, est_residual = model(mixture)
            loss, _ = or_pit_loss(est_speaker, est_residual, targets)
            
            # EVALUATION NAN SHIELD
            # If the dev audio is corrupted, skip it so it doesn't ruin the average
            if torch.isnan(loss) or torch.isinf(loss):
                continue
           
            
            # Flip the sign back to positive to get the actual dB score
            si_sdr_db = -1 * loss.item() 
            total_si_sdr += si_sdr_db
            valid_batches += 1
            
    # Prevent division by zero
    if valid_batches == 0:
        return 0.0
        
    avg_si_sdr = total_si_sdr / valid_batches
    return avg_si_sdr

def master_evaluation_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Evaluating on: {device}\n")
    
    model = MambaDeflationaryExtractor(d_model=256, d_state=16, d_conv=4, expand=2, num_blocks=4)
    
    
    
    FINAL_CHECKPOINT = "../weights/mamba_dexformer_epoch_115.pth"
    
    if not os.path.exists(FINAL_CHECKPOINT):
        print(f"Final checkpoint not found at: {FINAL_CHECKPOINT}")
        return
        
    print(f"Loading final model weights from {FINAL_CHECKPOINT}...")
    state_dict = torch.load(FINAL_CHECKPOINT, map_location=device)
    clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(clean_state_dict)
    model.to(device)
    
    
    ROOT_DATA_DIR = "../data/"
    DEV_FOLDERS = {
        2: "dev_mix2/kaggle/working/dev_mix2/",
        3: "dev_mix3/kaggle/working/dev_mix3/",
        4: "dev_mix4/kaggle/working/dev_mix4/",
        5: "dev_mix5/kaggle/working/dev_mix5/"
    }
    
  
    print("FINAL EVALUATION SCORES(115 EPOCH): ")
   

    for speakers in [2, 3, 4, 5]:
        data_path = os.path.join(ROOT_DATA_DIR, DEV_FOLDERS[speakers])
        
        if not os.path.exists(data_path):
            print(f"[{speakers} Speakers] Data not found at: {data_path}. Skipping.")
            continue
            
        val_dataset = CustomSpeechMixtureDataset(base_dir=data_path, num_speakers=speakers)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
        
        score = evaluate_validation_set(model, val_loader, device)
        print(f"[{speakers} Speakers] Average SI-SDR: {score:.2f} dB")

# Run the pipeline
master_evaluation_pipeline()